# 第18章 视频处理（跟随代码）

> 《OpenCV轻松入门：面向Python》（李立宗）第 18 章跟随代码。**全量收录原书代码（例 18.1～例 18.4，及属性读取实测、无摄像头写视频等独立演示）**，缺失的演示已按书补全，代码以网页笔记 docs/opencv/ch18.md 为准（含笔误修正与判空保护）。
> cv2.imshow 的视频窗口需在**本地 Python 环境**运行（弹窗显示，Notebook 服务内看不到窗口）；运行前请确认**工作目录为素材根目录**（本文件夹），本章素材为视频文件 **video1.mp4**（1280×720、30 fps、484 帧）。
> 例 18.1、例 18.3 涉及摄像头（`cv2.VideoCapture(0)`），必须接有真实摄像头才能运行；没有摄像头的环境会读到 ret=False，无法出画面。原书 viptrain.avi、vtest.avi 统一用 video1.mp4 顶替，代码逻辑除注明处外一字未改。


## 本章总览

视频信号是非常重要的视觉信息来源。视频由一系列图像构成，这一系列图像被称为帧（frame），帧以固定的时间间隔从视频中获取；获取（播放）帧的速度称为帧速率（frame rate），单位通常用"帧/秒"表示，即 FPS（Frames Per Second）。从视频中提取出独立的帧，就可以用图像处理的方法逐帧处理，达到处理视频的目的。全章例题：例 18.1（捕获摄像头视频）、例 18.2（播放视频文件）、例 18.3（保存摄像头视频）、例 18.4（逐帧 Canny 边缘检测）。

- **cv2.VideoCapture 类**：既管摄像头又管视频文件——构造参数是设备 ID 号就是摄像头（0 是第 1 个、-1 随机），是文件名就是视频文件；标准流程四步：构造 → isOpened() 检查 → read() 循环读帧 → release() 释放；
- **CAP_PROP_* 属性**：get(propId) 读、set(propId, value) 改，宽、高、帧速率、总帧数最常用，POS_FRAMES 可定位到任意帧；
- **grab / retrieve**：read() = grab() + retrieve()，用于同步一组多头摄像头（立体摄像头、Kinect）；
- **cv2.VideoWriter 类**：把帧序列按 fourcc 编码器、帧速率、帧尺寸写成视频文件，write() 逐帧写入、release() 之后文件才完整。

OpenCV 在各种操作系统中均支持 AVI 格式的视频文件。


## 18.1 VideoCapture 类

OpenCV 提供了 `cv2.VideoCapture` 类来处理视频，处理方式简单、快捷，既能处理视频文件，又能处理摄像头信息。OpenCV 官网在介绍它时特别强调：**视频处理完以后，要记得释放摄像头对象**（`cap.release()`）。

### 18.1.1 类函数介绍

| 函数 | 语法 | 说明 |
| --- | --- | --- |
| 初始化 | `cap = cv2.VideoCapture(0)` / `cap = cv2.VideoCapture('文件名')` | 参数是摄像头 ID 号（默认 -1 随机选；0 是第 1 个、1 是第 2 个……）则打开摄像头；参数是文件名则打开视频文件。部分平台 ID 为 -1 时会弹窗让用户手动选择 |
| isOpened() | `retval = cap.isOpened()` | 检查初始化（或 open）是否成功：True 成功、False 失败 |
| open() | `retval = cap.open(index)` / `cap.open(filename)` | 初始化失败时的补救打开，成功返回 True |
| read() | `retval, image = cap.read()` | 捕获一帧：retval 表示是否捕获成功，image 是捕获到的帧（没有帧被捕获时为空） |
| release() | `cap.release()` | 关闭摄像头/视频文件，释放对象 |
| get() / set() | `retval = cap.get(propId)` / `retval = cap.set(propId, value)` | 读取 / 设置属性 |

常用属性（CAP_PROP_*，完整 0～40 号属性表见网页笔记）：`CAP_PROP_POS_FRAMES`(1) 接下来要解码/捕获的帧索引（从 0 起）、`CAP_PROP_FRAME_WIDTH`(3) / `CAP_PROP_FRAME_HEIGHT`(4) 帧宽/高、`CAP_PROP_FPS`(5) 帧速率、`CAP_PROP_FOURCC`(6) 编码器、`CAP_PROP_FRAME_COUNT`(7) 总帧数。`cap.set(cv2.CAP_PROP_POS_FRAMES, n)` 能把读取位置直接跳到第 n 帧，配合 FRAME_COUNT 可以做"只处理前 100 帧""每隔 10 帧抽 1 帧"这类需求（部分后端按关键帧定位，可能有偏差，读完第一帧最好用 get 复核）。

视频时长 = FRAME_COUNT ÷ FPS：实测 video1.mp4 为 484 ÷ 30 ≈ 16.13 秒。写脚本处理视频前，先读一遍这四个属性，能避免"尺寸写错写不出文件""帧数算错没读全"这类低级错误。


### 属性读取实测：给 video1.mp4 做"体检"

用 `get()` 逐个读出 video1.mp4 的关键属性，fourcc 是整数，按 4 字节还原成字符。实测输出：

```text
isOpened: True
宽度: 1280.0
高度: 720.0
帧速率: 30.0
总帧数: 484.0
编码器: h264
```


In [ ]:
# ===== 18.1.1 属性读取实测：给 video1.mp4 做"体检" =====
import cv2

cap = cv2.VideoCapture('video1.mp4')              # 打开视频文件
print('isOpened:', cap.isOpened())                # True：成功打开
print('宽度:', cap.get(cv2.CAP_PROP_FRAME_WIDTH))     # 1280.0
print('高度:', cap.get(cv2.CAP_PROP_FRAME_HEIGHT))    # 720.0
print('帧速率:', cap.get(cv2.CAP_PROP_FPS))           # 30.0
print('总帧数:', cap.get(cv2.CAP_PROP_FRAME_COUNT))   # 484.0
f = int(cap.get(cv2.CAP_PROP_FOURCC))             # fourcc 是整数，按 4 字节还原成字符
print('编码器:', ''.join(chr((f >> 8 * i) & 0xFF) for i in range(4)))   # h264
cap.release()                                     # 用完释放


### grab 函数与 retrieve 函数

读取一个摄像头的视频数据，最简便的方法是 `read()`；但如果需要同步一组或一个多头（multihead）摄像头（例如立体摄像头或 Kinect）的视频数据，该函数就无法胜任了。可以把 `read()` 理解为由 `grab()` 和 `retrieve()` 组成：`grab()` 用来指向下一帧（成功时返回 True），`retrieve()` 用来解码并返回一帧（返回 (retval, image) 二元组，未成功 image 为空图像）。对于一组摄像头，先用每个摄像头各自 `grab()` 抓齐，再逐个 `retrieve()` 解码，就能拿到同一时刻的多路帧：

```python
# ===== 18.1.1 多摄像头同步捕获（原书示例） =====
success0 = cameraCapture0.grab()          # 摄像头 0 先指向下一帧
success1 = cameraCapture1.grab()          # 摄像头 1 先指向下一帧
if success0 and success1:                 # 两路都指向成功才解码
    frame0 = cameraCapture0.retrieve()[1] # retrieve 返回 (retval, 帧)，取第 2 个元素
    frame1 = cameraCapture1.retrieve()[1] # 两路帧来自同一时刻
```

与 VideoCapture 类内的其他函数一样，`grab()` 和 `retrieve()` 也能用来读取视频文件。

**原书两处笔误**：①原书 18.1.1 节写道"函数 v2.VideoCapture.grab() 捕获的视频帧"，漏了首字母 c，应为 cv2.VideoCapture.grab()；②原书多摄像头示例把 `retrieve()` 的返回值直接赋给单个变量，拿到的是 (retval, image) 二元组而不是帧——上面代码已改为 `[1]` 取出帧。


### 18.1.2 捕获摄像头视频 · 例 18.1

计算机视觉要处理的对象更多时候是从摄像设备中实时读入的视频流，OpenCV 通过 `cv2.VideoCapture` 类提供了非常方便的捕获摄像头视频（video capture）的方法。

**【例 18.1】使用 `cv2.VideoCapture` 类捕获摄像头视频**：参数 0 打开第 1 个摄像头，while 循环里逐帧 `read()` → `imshow()` 显示，按 ESC（ASCII 码 27）退出；`waitKey(1)` 每帧只停 1ms，画面连续。

**原书代码未判空**：循环体里没有检查 ret/frame——摄像头中途断开或视频文件读完后，`cap.read()` 返回 ret=False、frame=None，紧接着的 `cv2.imshow('frame', None)` 会抛出 `error: ... !_src.empty()`。稳妥写法是在 imshow 之前加 `if frame is None: break`（例 18.2、例 18.4 的跟随代码已加）。


In [2]:
# ===== 18.1.2 例 18.1：捕获并显示摄像头视频 =====
import numpy as np
import cv2

cap = cv2.VideoCapture(0)          # 打开第 1 个摄像头
while(cap.isOpened()):             # 只要摄像头处于打开状态就循环
    ret, frame = cap.read()        # 捕获一帧：ret 是否成功，frame 是帧
    cv2.imshow('frame', frame)     # 显示当前帧
    c = cv2.waitKey(1)             # 等 1ms，期间可按键
    if c == 27:                    # 27 是 ESC 键的 ASCII 码，按 ESC 退出
        break
cap.release()                      # 释放摄像头
cv2.destroyAllWindows()            # 关闭所有窗口


### 18.1.3 播放视频文件 · 例 18.2

播放视频文件时，把 `cv2.VideoCapture()` 的参数设置为视频文件的名称即可。播放时，可以通过设置 `cv2.waitKey()` 的参数值来控制每一帧的持续（停留）时间：参数较小 → 每帧停留时间短，播放速度快；参数较大 → 播放速度慢。该参数的单位是 ms，通常设置为 25 即可。

**【例 18.2】使用 `cv2.VideoCapture` 类播放视频文件**：video1.mp4 实测循环逐帧读取共读到 484 帧，与 CAP_PROP_FRAME_COUNT 属性报告的 484 一致。waitKey(25) 的播放速度只取决于停留时间，与文件自身 30 fps 的帧速率不必相同——想按原始速度播放，取值应约为 `int(1000 / FPS)`。


In [5]:
# ===== 18.1.3 例 18.2：播放视频文件 =====
import numpy as np
import cv2

cap = cv2.VideoCapture('video1.mp4')   # 参数是文件名 → 播放视频文件（原书 viptrain.avi）
while(cap.isOpened()):                 # 只要视频处于打开状态就循环
    ret, frame = cap.read()            # 依次读取每一帧
    if frame is None:                  # 读不到帧（视频播完）就退出（原书缺此判断）
        break
    cv2.imshow('frame', frame)         # 显示当前帧
    c = cv2.waitKey(25)                # 每帧停留 25ms，控制播放速度
    if c == 27:                        # 按 ESC 退出
        break
cap.release()                          # 释放视频对象
cv2.destroyAllWindows()                # 关闭所有窗口


## 18.2 VideoWriter 类

OpenCV 中的 `cv2.VideoWriter` 类可以将图片序列保存成视频文件，也可以修改视频的各种属性，还可以完成对视频类型的转换。

### 18.2.1 类函数介绍

**构造函数**用于完成初始化工作：

```python
# ===== 18.2.1 构造 VideoWriter：两步走 =====
fourcc = cv2.VideoWriter_fourcc(*'XVID')                # 先定编码器：*'XVID' 拆成 4 个字符
out = cv2.VideoWriter('output.avi', fourcc, 20, (1024, 768))  # 文件名、编码器、帧速率、帧尺寸
```

- **filename**：输出目标视频的存放路径和文件名，文件已存在则覆盖；
- **fourcc**：视频编/解码类型。OpenCV 用 `cv2.VideoWriter_fourcc()` 指定，4 个字符参数构成编/解码器的"4 字标记"（fourcc, four-character code）。常用：`('I','4','2','0')` 未压缩 YUV 4:2:0（.avi，文件较大、兼容性好）、`('P','I','M','I')` MPEG-1（.avi）、`('X','V','I','D')` MPEG-4（.avi，大小适中）、`('T','H','E','O')` Ogg 编码（.ogv）、`('F','L','V','I')` Flash 视频（.flv）。更多组合可在 fourcc.org 查询。若 fourcc 为 -1，程序运行时会弹出"视频压缩"对话框，由用户选择压缩程序和压缩质量：

```python
fourcc = -1                                             # 运行时弹出压缩程序选择对话框
out = cv2.VideoWriter('output.avi', fourcc, 20, (1024, 768))
```

- **fps**：帧速率；
- **frameSize**：每一帧的长和宽，写法是 (宽, 高)——注意与 NumPy 数组"行（高）在前"的方向相反；
- **isColor**：是否为彩色图像，写灰度视频要设 False。

**原书笔误**：原书写"('T','H','E','O') 表示 Ogg Vorbis 编码类型"——THEO 实际对应 **Theora**（Ogg 项目开放的视频编解码器）；Ogg Vorbis 是 Ogg 容器里的**音频**编码格式。写 .ogv 视频用 THEO 没错，但名字不应混用。

**容器与编解码器是两回事**：.avi、.mp4、.ogv 这类扩展名指的是容器（container，装帧和音轨的"包装盒"），fourcc 指定的才是编解码器（codec，压缩帧的"压缩方法"）。OpenCV 在各操作系统上都支持 AVI 容器；跨平台写视频拿不准时，.avi + MJPG/XVID 是最稳的组合。

**write 函数**：`out.write(image)` 用于写入下一帧视频，image 通常要求为 BGR 模式的彩色图像帧。**release 函数**：`out.release()` 释放对象——不释放，视频文件不完整。


### 18.2.2 保存视频 · 例 18.3

保存视频包括创建对象、写入视频、释放对象等多个步骤：①创建对象——文件名（如 'out.avi'）、`fourcc = cv2.VideoWriter_fourcc(*'XVID')`、帧速率（如 20）、帧尺寸（如 (640, 480)），参数可先设好再传入，也可直接写进构造函数；②`out.write(frame)` 逐帧写入读取到的视频帧；③完成写入后 `out.release()` 释放对象。

**【例 18.3】使用 `cv2.VideoWriter` 类保存摄像头视频文件**：边捕获边保存——read 到一帧就 out.write(frame) 写入 output.avi 并 imshow 显示，按 ESC 退出，程序运行后当前目录下即得到 output.avi。


In [7]:
# ===== 18.2.2 例 18.3：边捕获边保存摄像头视频 =====
import numpy as np
import cv2

cap = cv2.VideoCapture(0)                              # 打开摄像头
fourcc = cv2.VideoWriter_fourcc('I', '4', '2', '0')    # 未压缩 YUV 4:2:0 编码
out = cv2.VideoWriter('output.avi', fourcc, 20, (640, 480))  # 输出文件、20fps、640×480
while(cap.isOpened()):
    ret, frame = cap.read()                            # 捕获一帧
    if ret == True:                                    # 捕获成功才处理
        out.write(frame)                               # 写入输出视频
        cv2.imshow('frame', frame)                     # 同时显示
        if cv2.waitKey(1) == 27:                       # 按 ESC 退出
            break
    else:
        break
cap.release()                                          # 释放摄像头
out.release()                                          # 释放写视频对象（不释放文件不完整）
cv2.destroyAllWindows()                                # 关闭所有窗口


**补充演示**（网页笔记实跑，增补）：没有摄像头也能把 VideoWriter 跑通——用 NumPy 造 30 帧纯色帧写进去，再读回验证帧数。注意 `frameSize` 写 (160, 120) 是宽在前，而造帧时 `np.zeros((120, 160, 3))` 是高（行数）在前——两个参数方向相反，写反了帧就会变形。实测写进去 30 帧、读回来还是 30 帧，帧数完全一致；第 0/1/2 帧依次为蓝、绿、红，颜色轮换清晰可见——写进去什么颜色，读出来还是什么颜色。实测输出：

```text
writer isOpened: True
读回总帧数: 30.0
读回逐帧计数: 30
```


In [ ]:
# ===== 18.2.2 无摄像头版：造 30 帧写视频再读回验证 =====
import numpy as np
import cv2

fourcc = cv2.VideoWriter_fourcc(*'XVID')        # MPEG-4 编码，大小适中
out = cv2.VideoWriter('test_out.avi', fourcc, 10, (160, 120))  # 160×120、10fps
print('writer isOpened:', out.isOpened())       # True：输出文件已就绪
for i in range(30):                             # 造 30 帧
    img = np.zeros((120, 160, 3), dtype=np.uint8)   # 黑底帧，注意形状是 (高, 宽, 3)
    img[:, :, i % 3] = 255                      # 每帧点亮一个通道，颜色轮换
    out.write(img)                              # 逐帧写入
out.release()                                   # 释放后文件才完整

back = cv2.VideoCapture('test_out.avi')         # 读回刚写的文件
print('读回总帧数:', back.get(cv2.CAP_PROP_FRAME_COUNT))   # 30.0
n = 0
while back.isOpened():                          # 再逐帧数一遍
    ret, _ = back.read()
    if not ret:
        break
    n += 1
back.release()
print('读回逐帧计数:', n)                        # 30


## 18.3 视频操作基础 · 例 18.4

视频是由视频帧构成的，将视频帧从视频中提取出来，对其使用图像处理的方法进行处理，就可以达到处理视频的目的。

**【例 18.4】提取视频的 Canny 边缘检测结果**：把例 18.2 的播放循环当骨架，在 `read()` 之后、`imshow()` 之前插入 `frame = cv2.Canny(frame, 100, 200)`，播放的就是边缘检测视频——video1.mp4 实测 484 帧全部通过 Canny 处理，无一帧报错。套路就此固定：**插入任意图像处理函数即完成逐帧处理**，换成 `cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)` 是逐帧灰度化，换成第 8 章的 `cv2.erode()` 就是逐帧腐蚀——插入的函数不同，得到的"处理视频"风格就不同。


In [10]:
# ===== 18.3 例 18.4：逐帧做 Canny 边缘检测 =====
import numpy as np
import cv2

cap = cv2.VideoCapture('video1.mp4')     # 打开视频文件（原书 viptrain.avi）
while(cap.isOpened()):
    ret, frame = cap.read()              # 读一帧
    if frame is None:                    # 读不到帧就退出（原书缺此判断，见下）
        break
    frame = cv2.Canny(frame, 100, 200)   # 对当前帧做 Canny 边缘检测
    cv2.imshow('frame', frame)           # 显示处理结果
    c = cv2.waitKey(1)                   # 每帧停 1ms
    if c == 27:                          # 按 ESC 退出
        break
cap.release()                            # 释放
cv2.destroyAllWindows()                  # 关闭所有窗口


**补充演示**（跟随代码原有拓展）：帧差法找运动目标——对视频帧做差分，静止的背景会抵消，运动的物体会留下来。简化版：先灰度化，再用 `cv2.subtract(gray, cv2.erode(gray, kernel))` 做腐蚀后差分，得到近似前景的边缘。更正规的做法是相邻帧相减（两帧时间上够近，背景不动、目标移动）或背景建模——把一段背景学习成模型再减除，对应第 17 章 GrabCut 前景提取的思路。


In [18]:
# ===== 18.3 补充演示：帧差法（腐蚀后差分，跟随代码原有拓展） =====
cap=cv2.VideoCapture(0) #打开摄像头

while(cap.isOpened()):
    ret,frame=cap.read() #读取一帧图像
    if frame is None:
        break
    if ret is True:
        gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY) #转换为灰度图
        kernel=np.ones((5,5),np.uint8) #定义卷积核
        erosion=cv2.erode(gray,kernel,iterations=1) #腐蚀
        b=cv2.subtract(gray,erosion) #差分
        cv2.imshow('frame',b) #显示图像
        if cv2.waitKey(1)==27:
            break

cap.release()
cv2.destroyAllWindows()


## 坑与陷阱（速查）

1. **不判空就处理帧**：视频播完、文件损坏或摄像头断开时 `cap.read()` 返回 ret=False、frame=None，紧接着 imshow(None) 或 Canny(None) 直接抛 `!_src.empty()`——原书例 18.1、例 18.4 都没判，照抄会在结尾崩一次；
2. **忘写 `out.release()`**：VideoWriter 不释放，视频文件头没写完——打不开、能打开但帧数为 0 或只有一部分；`cap.release()` 同理，摄像头不释放会被其他程序"占用中"；
3. **fourcc 与扩展名不匹配**：XVID 配 .mp4、mp4v 配 .avi 经常导致 `out.isOpened()` 为 False、write() 静默丢帧——成套使用：.avi + XVID/MJPG/I420，.mp4 + mp4v；
4. **frameSize 与实际帧尺寸不一致**：构造函数是宽在前，数组形状是行（高）在前，方向相反——不一致时部分后端自动缩放，部分直接写失败；
5. **isColor 与帧通道数不匹配**：写灰度帧要 isColor=False；读回灰度视频时 OpenCV 仍按 3 通道 BGR 返回；
6. **read/retrieve 是二元组**：`frame = cap.read()` 拿到的是 (retval, image)，imshow 直接报错，必须 `ret, frame = cap.read()`；
7. **waitKey 的参数决定播放节奏**：文件播放常取 25，摄像头实时显示取 1；写 0 表示永久等待按键——画面会"卡住"不动；
8. **摄像头 ID 与文件名混淆**：VideoCapture(0) 打开第 1 个摄像头；VideoCapture('0') 把字符串当文件名，打不开返回 False。
